In [ ]:
import numpy as np
from typing import *

def prune_by_magnitude(W, sparsity):
    # 按绝对值大小剪枝: 值越小越不重要
    thr = np.percentile(np.abs(W), sparsity*100)
    mask = (np.abs(W) > thr).astype(np.float32)
    return mask, W * mask

def structured_prune(W, sparsity, dim=0):
    # 结构化剪枝: 以整个通道为单位, L2 范数衡量重要性
    imp = np.linalg.norm(W, ord=2, axis=dim)
    thr = np.percentile(imp, sparsity*100)
    m = (imp > thr).astype(np.float32)
    if dim == 0: m = m[:, None] else: m = m[None, :]
    return m, W * m

def random_prune(W, sparsity):
    # 随机剪枝: baseline, 用来对比验证重要性准则是否有效
    m = np.random.binomial(1, 1-sparsity, size=W.shape).astype(np.float32)
    return m, W * m

def prune_by_std(W, sparsity):
    # Z-score 剪枝: 偏离均值小的权重贡献低
    z = np.abs((W - np.mean(W))) / (np.std(W) + 1e-12)
    thr = np.percentile(z, sparsity*100)
    m = (z > thr).astype(np.float32)
    return m, W * m

def print_stats(orig, pruned):
    s = np.sum(pruned==0)/pruned.size
    nonz = np.count_nonzero(pruned)
    print(f'  nonzeros: {np.count_nonzero(orig)} -> {nonz}  ({nonz/np.count_nonzero(orig)*100:.1f}%)')
    print(f'  sparsity: {s*100:.1f}%')


In [ ]:
if __name__ == '__main__':
    np.random.seed(42)
    W = np.random.randn(8, 16) * 0.5

    print('1. Random pruning (sparsity=70%)')
    _, Wr = random_prune(W, .7); print_stats(W, Wr)

    print('\n2. Magnitude pruning (sparsity=70%)')
    _, Wm = prune_by_magnitude(W, .7); print_stats(W, Wm)

    print('\n3. Structured pruning (dim=0, sparsity=50%)')
    ms, Ws = structured_prune(W, .5, 0)
    print_stats(W, Ws)
    print(f'  alive channels: {int(np.sum(ms[:,0]))}/{W.shape[0]}')

    print('\n4. Sparsity vs MSE curve')
    for sp in [0., .3, .5, .7, .9, .95]:
        _, Wp = prune_by_magnitude(W, sp)
        print(f'  sparsity {sp*100:4.0f}% -> MSE = {np.mean((W-Wp)**2):.6f}')

    print('\n5. Importance criteria comparison (sparsity=80%)')
    for name, fn in [('Random', random_prune), ('Magnitude', prune_by_magnitude), ('Z-score', prune_by_std)]:
        _, Wp = fn(W, .8)
        print(f'  {name:>10}: MSE = {np.mean((W-Wp)**2):.6f}')
